# Derivatives

JAX can differentiate the same function used to evaluate a light curve.
Here we compute a parallax light curve and its derivatives with respect
to the event parameters, both parallax components, and the linear limb
coefficient `a1`.


In [ ]:
%matplotlib inline
from pathlib import Path
import sys
root = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
            if (p / "pyproject.toml").exists())
sys.path.insert(0, str(root))
from example._common import style
style()


In [ ]:
import jax

jax.config.update("jax_enable_x64", True)
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt
import singlens as sl


## Define a parameterized light curve

Keep observation times and sky geometry fixed. Each column of the
Jacobian gives $\partial A(t_i)/\partial\theta_j$: the change in
magnification per unit change in that parameter.


In [ ]:
times = 8000.0 + jnp.linspace(-40.0, 40.0, 1001)
geometry = sl.AnnualParallax(ra_deg=266.0, dec_deg=-29.0, t_ref=8000.0)
# theta = [t0, u0, t_e, rho, pi_e_n, pi_e_e, a1]
theta = jnp.array([8000.0, 0.05, 80.0, 0.1, 0.2, -0.1, 0.5])


def curve(theta, *, source="linear"):
    return sl.magnification(
        times,
        *theta[:3],
        rho=theta[3],
        a1=theta[6] if source == "linear" else None,
        parallax=geometry,
        pi_e_n=theta[4],
        pi_e_e=theta[5],
    )


A = jax.jit(curve)(theta)
jacobian = jax.jit(jax.jacfwd(curve))(theta)
print(jacobian.shape)  # (observation times, parameters)


Compare uniform and limb-darkened sources at identical geometry. The
uniform curve has no `a1` dependence, so its derivative with respect
to `a1` is zero. That panel shows only the linear profile.


In [ ]:
from matplotlib.ticker import MaxNLocator


def uniform_curve(p):
    return curve(p, source="uniform")


A_uniform = jax.jit(uniform_curve)(theta)
jacobian_uniform = jax.jit(jax.jacfwd(uniform_curve))(theta)
columns = [0, 2, 1, 3, 6, 4, 5]
labels = [r"$t_0$", r"$t_E$", r"$u_0$", r"$\rho$", r"$a_1$", r"$\pi_{E,N}$", r"$\pi_{E,E}$"]
fig, axes = plt.subplots(
    8,
    1,
    figsize=(10, 15),
    sharex=True,
    layout="constrained",
    gridspec_kw={"height_ratios": [2.4] + [1] * 7},
)
for name, values, derivatives in [
    ("Uniform", A_uniform, jacobian_uniform),
    ("Linear (a₁ = 0.5)", A, jacobian),
]:
    (line,) = axes[0].plot(times - 8000.0, values, label=name)
    for ax, column in zip(axes[1:], columns):
        if name == "Uniform" and column == 6:
            continue
        ax.plot(times - 8000.0, derivatives[:, column], color=line.get_color(), lw=1.1)
axes[0].set_ylabel("Magnification")
for ax, label in zip(axes[1:], labels):
    ax.set_ylabel(r"$\partial A/\partial$ " + label)
    ax.axhline(0.0, color="0.7", lw=0.6)
    ax.yaxis.set_major_locator(MaxNLocator(nbins=3))
    ax.ticklabel_format(axis="y", style="sci", scilimits=(-2, 3), useMathText=True)
axes[-1].set(xlim=(-40, 40), xlabel="HJD − 2458000 [day]")
fig.legend(*axes[0].get_legend_handles_labels(), loc="outside upper right", ncol=2)
fig.align_ylabels(axes)
plt.show()


Changes in `t0` mostly shift the curve in time; `u0`, `rho`, and `a1`
change its peak shape. The parallax derivatives describe the response
to the north/east projected Earth motion at the fixed reference epoch.
Derivative amplitudes alone do not quantify parameter constraints, which
also depend on noise, cadence, and correlations with other parameters.

## Scalar objectives

For a scalar loss, `grad` computes a parameter gradient without returning
the full time-by-parameter Jacobian. As a simple example, profile the
source and blend fluxes at each event parameter vector:


In [ ]:
observed = 1.2 * A + 0.15
error = 0.01 + 0.002 * observed


def loss(p):
    return sl.fit_flux(curve(p), observed, error).chi2


gradient = jax.jit(jax.grad(loss))(theta)
print(gradient.shape)


This noiseless example has its minimum at the input event, so its gradient
is close to zero there. For a Bayesian posterior, include flux parameters
and priors as shown in [inference](inference.ipynb).

## Independent events

Use `vmap` to apply the same light-curve function to multiple parameter
vectors. The first output axis indexes events, and the second indexes
observation times.


In [ ]:
events = jnp.stack([theta, theta.at[1].set(0.07)])
curves = jax.jit(jax.vmap(curve))(events)
print(curves.shape)


For named derivatives, `jax.jacfwd(model.magnification, argnums=1)` accepts
a `Parameters` object and returns fields such as `.rho`, `.a1`, and
`.pi_e_n`. Automatic differentiation follows the numerical approximation;
see [numerical accuracy](../numerics.md) when derivatives determine a
precision fit, especially near source contact.
